# ¿Qué detector elegirías?

Comparen Faster R-CNN, YOLOv8n y YOLO-World con tres imágenes sintéticas proporcionadas. Después construyan una variante de R-CNN: generen regiones candidatas, extraigan sus características con ResNet-18 congelada y entrenen una cabeza de clasificación y regresión de cajas.

Se permite apoyo de IA pero cada integrante debe explicar las decisiones y los resultados.

## Preparación

Ejecuten en Google Colab. La sesión necesita Internet para descargar pesos y datos. Se puede usar CPU, aunque la extracción por regiones puede ser lenta. Una GPU acelera esa etapa; su disponibilidad en google colab no está garantizada.

In [ ]:
%pip -q install ultralytics opencv-contrib-python-headless matplotlib pillow

import io
import time
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from PIL import Image
from google.colab import files
from ultralytics import YOLO, YOLOWorld

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE)

## Imágenes

Se utilizan tres imágenes sintéticas: una escena con persona, perro y automóvil; una cafetería con objetos parcialmente ocultos; y una persona con casco. Descárguenlas desde el repositorio para que todos comparen los mismos casos. No tienen anotaciones de evaluación y no garantizan detecciones correctas.

In [ ]:
from urllib.request import urlopen, Request

BASE_URL = 'https://raw.githubusercontent.com/noe-tec/machine-learning/main/datasets/object_detection_comparison/'
IMAGE_NAMES = ['01_escena_cotidiana.png', '02_oclusiones.png', '03_casco.png']
images = []
for name in IMAGE_NAMES:
    try:
        request = Request(BASE_URL + name, headers={'User-Agent': 'object-detection-lesson'})
        with urlopen(request, timeout=60) as response:
            raw = response.read()
        image = Image.open(io.BytesIO(raw)).convert('RGB')
        image.thumbnail((640, 640))
        images.append((name, image.copy()))
    except Exception as exc:
        raise RuntimeError(f'No se pudo cargar {name}. Compruebe la conexión y que las imágenes estén publicadas en main.') from exc
for name, image in images:
    plt.figure(figsize=(7, 5))
    plt.imshow(image)
    plt.title(name)
    plt.axis('off')
    plt.show()

## Faster R-CNN

Carguen el detector preentrenado `fasterrcnn_mobilenet_v3_large_320_fpn` de torchvision. Ejecuten inferencia sobre las tres imágenes de muestra, dibujen las cajas con sus clases y puntuaciones y registren el tiempo por imagen. Elijan y justifiquen un umbral de confianza. Identifiquen un acierto y un error u omisión.

### Prueba con una imagen propia: Faster R-CNN

Carguen una imagen propia o autorizada, ejecuten el detector y expliquen un resultado observado.

## YOLO con clases predefinidas

Carguen YOLOv8n con los pesos `yolov8n.pt` y consulten su catálogo de clases. Ejecuten detección sobre las mismas tres imágenes y registren resultados y tiempos. Expliquen si el catálogo incluye el casco y qué implica esa limitación.

### Prueba con una imagen propia: YOLOv8n

Carguen una imagen propia o autorizada. Ejecuten el modelo e interpreten sus detecciones considerando las clases disponibles.

## YOLO-World

Carguen `yolov8s-worldv2.pt` y definan un vocabulario textual para las imágenes de muestra. Ejecuten inferencia y registren resultados y tiempos. Después añadan `helmet` u otra clase ausente del catálogo de YOLOv8n y comparen ambos modelos sobre la imagen correspondiente. Expliquen qué cambia sin reentrenar y comprueben si la nueva detección es correcta.

### Prueba con una imagen propia: YOLO-World

Carguen una imagen propia o autorizada y elijan las clases que desean buscar. Prueben dos vocabularios y expliquen los cambios observados.

## Comparación

Construyan una tabla con imagen, modelo, vocabulario utilizado, umbral, tiempo y número de detecciones. Distingan las ejecuciones de YOLO-World con vocabularios diferentes. Excluyan las descargas de la medición y describan qué operaciones están cronometrando.

## Construcción de R-CNN

Construyan una variante de R-CNN con propuestas de Selective Search, ResNet-18 congelada y una cabeza neuronal para clasificación y regresión de cajas. Extraigan las características en su sesión y documenten el costo. Se admite avance parcial; conserven evidencia de cada etapa conseguida.

### Datos y propuestas

Descarguen PASCAL VOC 2007 mediante torchvision y seleccionen un subconjunto pequeño con persona, automóvil y perro. Generen propuestas con Selective Search y asignen etiquetas según IoU: objeto con IoU ≥ 0.5, fondo con IoU < 0.3 e ignorar los casos intermedios. Verifiquen que haya ejemplos de las tres clases y de fondo. Pueden añadir cajas anotadas y perturbaciones para obtener suficientes positivos; expliquen esta decisión.

### Extracción de características

Carguen ResNet-18 con pesos `IMAGENET1K_V1`, retiren su capa de clasificación y congelen sus parámetros. Apliquen las transformaciones de esos pesos a cada recorte y extraigan vectores de 512 características en lotes.

Midan por separado la descarga, la generación de propuestas y la extracción. Registren imágenes y regiones procesadas. Expliquen por qué congelar la red no elimina el tiempo de extracción y cómo cambiaría el costo al ampliar el subconjunto.

### Clasificación y regresión de cajas

Filtren las regiones ambiguas y normalicen las características conservando la media y desviación del entrenamiento. Implementen una cabeza para dos salidas: cuatro clases (fondo, persona, automóvil y perro) y cuatro ajustes de caja.

Definan la codificación de los ajustes relativos, construyan el ciclo de entrenamiento y utilicen entropía cruzada para clasificación y Smooth L1 únicamente en positivos para regresión. Registren la pérdida por época. La pérdida de entrenamiento no evalúa el detector en imágenes nuevas.

### Supresión de cajas redundantes

Implementen NMS utilizando `torchvision.ops.nms`.

### Prueba con una imagen propia: R-CNN

Como extensión, carguen una imagen propia o autorizada con persona, automóvil o perro. Generen propuestas y extraigan características con las mismas transformaciones. Apliquen la normalización del entrenamiento, ejecuten la cabeza, decodifiquen los ajustes, limiten las cajas al tamaño de la imagen y apliquen NMS por clase. Muestren las detecciones y midan el tiempo total.

## Entrega

Entregar la libreta resultante.